# Advanced: Instantiation Options

In [ ]:
# Import all of the packages we will use throughout this notebook
import numpy as np
import pandas as pd
import lkdata as ld

In [ ]:
# Optional display options
np.set_printoptions(threshold=10)  # limiting to printing up to 10 values at a time
pd.set_option("display.max_columns", 15)  # limiting to printing up to 15 columns

In [ ]:
ntime = 120

# Generate times and random corrections
times = np.linspace(0, 24, ntime)
time_corr = np.random.random(times.shape)

# Generate some random data
series_data = np.random.standard_normal(times.shape)

# 1 - Time
- via `time_indices` (recommended)
  - a dictionary of time arrays
- via `index`, a la `pandas`
  - arraylike
  - pandas index object
- None or via `ntime` - automatically generates a RangeIndex called "time_index". Specifying `ntime` should be unnecessary.

**Notes:**
- A ranged index called "time_index" is automatically created unless an index with the same name is given.
- Duplicate indices are dropped, defaulting to the first in order given.

## 1.1 - Recommended: `time_indices`

It is easy to give any number of named indices via dictionary with a `str: array-like` pair

In [ ]:
series = ld.DataSeries(
    series_data, time_indices={"fake_jd": times, "corrected_time": times + time_corr}
)
series.describe_series()

## 1.2 - Array-like via `index`

This is the most Pandas-like and is easy for giving a single array for an index. It is automatically named "given_index" and the ranged "time_index" is created.


In [ ]:
series = ld.DataSeries(series_data, index=times)
series.describe_series()

## 1.3 - Pandas `Index` via `index`
This is the easiest when working with an existing pandas object with an index property (which includes the lkdata objects). Index names carry into the new index. "time_index" is created unless it exists in the given index.

### 1.3.1 - Using an index from an existing object

In [ ]:
df = pd.DataFrame(series_data, index=times)
df.index.name = "time_index"
series = ld.DataSeries(series_data, index=df.index)
series.describe_series()

### 1.3.2 - Creating an index beforehand
It's also possible to generate pandas indices directly, see the [pandas Index objects documentation](https://pandas.pydata.org/docs/reference/indexing.html) for all options.
This enables very fine-grained control over the indices of lkdata objects.

In [ ]:
index = pd.TimedeltaIndex(pd.to_timedelta(times, unit="d"), name="time")
series = ld.DataSeries(series_data, index=index)
# time_indices can also accept pd.Index objects, but will be renamed to the given key
# series = lk.DataSeries(series_data, time_indices={"time_diffname": index})
series.describe_series()

For multiple indices, see [pandas.MultiIndex](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.html).

In [ ]:
index = pd.MultiIndex.from_arrays(
    [times, times + time_corr], names=["custom", "custom_corrected"]
)

# Another example, combining multiple pandas Index objects. Note that names carry through:
# rindex = pd.RangeIndex(ntime, name="time_index")
# dtindex = pd.TimedeltaIndex(pd.to_timedelta(times, unit='d'), name="time")
# index = pd.MultiIndex.from_arrays([rindex, dtindex])

series = ld.DataSeries(
    series_data, index=index
)  # time_indices cannot accept a MultiIndex

series.describe_series()

# 2 - Space
- [2.1](#21---recommended-give-dictionaries-to-row_indices-and-col_indices) - `row_indices` and `col_indices` (recommended)
  - Values can be a list of unique ordered indices, with lengths `nrow` and `ncol` respectively (`cube` only), or
  - a full list of coordinate values
    - each with length `nrow`$\times$`ncol` for `cube` objects (defined explicitly or based on the shape of the data)
    - any length for `SeriesCollection`, lengths must match and be equal to the number of series present
- `nrow` and `ncol` - automatically generates a `RangeIndex` for each. (`Cube` only with flattened data)
- `columns` (`Cube` and `SeriesCollection`)
  -  For `Cube`, this will only work properly if "row" and "col" are in the column names to properly order data in the array representation.
  -  For `SeriesCollection` no interpretation of the columns will be attempted.
- None
  - for `Cube` generates `RangeIndex` if data is given as an (ntime, nrow, ncol) `ArrayLike`, otherwise raises an error where the shape cannot be interpreted as a cube.
  - generates a `RangeIndex` for the number or series present.

In [ ]:
ntime = 120
nrow = 10
ncol = 12

# Generate times and random corrections
times = np.linspace(0, 24, ntime)

# Generate some random data
data = np.random.standard_normal((ntime, nrow, ncol))

## 2.1 - Recommended: give dictionaries to `row_indices` and `col_indices`
Give indices for rows and columns via `row_indices` and `col_indices`.
Multiple row and column levels can be defined with different coordinate systems.
There are no naming requirements (save for a few reserved names) when provided this way as rows and column information is provided and stored separately.

### 2.1.1 - Arrays given as ranges of unique values
Useful where a coordinate system or a pixel reference point is given, rather than coordinates.

In [ ]:
# Arrays given as ranges of unique values
# Useful where only a reference pixel position is given
row_arr = np.arange(100, 100 + nrow)
col_arr = np.arange(200, 200 + ncol)

row_pos = row_arr * 1.8 - 13.5  # fake coordinate transformation
col_pos = col_arr * 0.8 - 1.35  # fake coordinate transformation
row_arr, col_arr

In [ ]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
    row_indices={"pix_row": row_arr, "row_pos": row_pos},
    col_indices={"pix_col": col_arr, "col_pos": col_pos},
)
pd.DataFrame(cube)

### 2.1.2 - Arrays given as coordinates

In [ ]:
row_arr = np.repeat(np.arange(100, 100 + nrow), ncol)
col_arr = np.tile(np.arange(200, 200 + ncol), nrow)

row_pos = row_arr * 1.8 - 13.5  # fake coordinate transformation
col_pos = col_arr * 0.8 - 1.35  # fake coordinate transformation
np.array(list(zip(row_arr, col_arr)))

In [ ]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
    row_indices={"pix_row": row_arr, "row_pos": row_pos},
    col_indices={"pix_col": col_arr, "col_pos": col_pos},
)
pd.DataFrame(cube)

In [ ]:
series_collection = ld.DataSeriesCollection(
    data.reshape(120, 120),
    row_indices={"row": row_arr},
    col_indices={"col": col_arr},
)
series_collection

## 2.2 - If coordinates aren't particularly important
If the data is given as an array with shape (ntime, nrow, ncol) and coordinates are not particularly important,
row and column information can be omitted and indices will be generated for both along with a `RangeIndex` to keep
track of each series.

In [ ]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
)
pd.DataFrame(cube)

In [ ]:
series_collection = ld.DataSeriesCollection(
    data.reshape(120, 120),
)
series_collection

If data is given as a flattened array, but the dimensions are known, specify nrow and ncol if a `RangeIndex` is sufficient.

In [ ]:
flat_data = np.random.standard_normal((ntime, nrow * ncol))
flat_data

In [ ]:
cube = ld.DataCube(
    flat_data,
    time_indices={"time": times},
    nrow=nrow,
    ncol=ncol,
)
pd.DataFrame(cube)

## 2.3 - From a pandas Index object


In [ ]:
# Indices as coordinates
row_arr = np.repeat(np.arange(100, 100 + nrow), ncol)
row_index = pd.Index(row_arr, name="row_index")
col_arr = np.tile(np.arange(200, 200 + ncol), nrow)
col_index = pd.Index(col_arr, name="col_index")
series_range = pd.RangeIndex(nrow * ncol, name="series")
columns = pd.MultiIndex.from_arrays([series_range, row_index, col_index])
np.array(list(zip(row_index, col_index)))

In [ ]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
    columns=columns,
)
pd.DataFrame(cube)

## 2.4 - Combining `columns`, `row_indices`, and `col_indices`
Using all available keywords is supported and an easy way to add new column indices to an existing set of columns while keeping track of which pertain to rows and columns. Again, though, the given `columns` need to have the string "row" or "col" in their names to be interpreted correctly as rows and columns for `Cube` objects.

In [ ]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
    row_indices={"row_pos": row_pos},
    col_indices={"col_pos": col_pos},
    columns=columns,
)
pd.DataFrame(cube)

# 3 - `Cube` from a pandas DataFrame

It may be convenient to create a pandas `DataFrame` at first to wrangle data and set indices and columns. It is possible to create a `Cube` from a pandas `DataFrame` using the `from_pandas` class method.

In [ ]:
df = pd.DataFrame(series_data, index=times)
df.index.name = "time_index"

In [ ]:
# Indices as coordinates
row_arr = np.repeat(np.arange(100, 100 + nrow), ncol)
row_index = pd.Index(row_arr, name="row_index")
col_arr = np.tile(np.arange(200, 200 + ncol), nrow)
col_index = pd.Index(col_arr, name="col_index")
series_range = pd.RangeIndex(nrow * ncol, name="series")
columns = pd.MultiIndex.from_arrays([series_range, row_index, col_index])

df = pd.DataFrame(data.reshape(120, 120), index=times, columns=columns)
df.index.name = "times"
df

In [ ]:
cube_df = ld.DataCube.from_pandas(
    df,
    nrow=10,  # DataFrames are 2D, nrow and ncol must be specified
    ncol=12,
    # uncertainty=data_err,  # DataFrames have no standard way to store uncertainty, if desired it must be given
    # **kwargs  # any additional metadata to provide
)
cube_df

In [ ]:
cube_df.describe_cube()